# Aula 17 — Sub-grafos: grafos dentro de grafos

**Objetivo:** compor um grafo pai que chama filhos compilados como nós — e
entender o que o filho lê, o que ele devolve e como isolar a interface.

**Pré-requisitos:** Aula 08 (reducers) e Aula 09 (schemas de interface).

**Como usar:** rode em ordem. Autocontido, 100% offline (sem LLM, sem chave).
Comece pela CÉLULA 0.

In [ ]:
# CÉLULA 0 — instalar dependências
# No Colab esta célula é necessária. Localmente (uv sync feito) pode pular.
%pip install -q langgraph
print('dependências prontas.')

## 1. Analogia: salão, cozinha e bar

O **salão** recebe o pedido e despacha para **cozinha** e **bar** (duas equipes
com rotinas próprias); depois **fecha a conta** juntando os dois resultados.

| Restaurante | No LangGraph | O que é de verdade |
|---|---|---|
| equipe com rotina própria | **sub-grafo compilado** | `filho.compile()` usado como nó |
| despachar o pedido | **`add_node('cozinha', filho)`** | o pai entrega o estado; o filho filtra o que lê |
| devolver só o pronto | **`output_schema`** | só essas chaves voltam ao pai |
| caderno interno da cozinha | **chave privada do filho** | nunca vaza para o pai |

A regra de bolso: o filho **lê** o que reconhece do pai e **devolve o estado
final completo** — inclusive ecoando a entrada. É esse eco que morde.

In [ ]:
# CÉLULA 1 — pai + 2 filhos (rode e veja o eco)
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class Cozinha(TypedDict):
    pedido: str
    prato: str

bc = StateGraph(Cozinha)
bc.add_node('cozinhar', lambda s: {'prato': f"prato({s['pedido']})"})
bc.add_edge(START, 'cozinhar'); bc.add_edge('cozinhar', END)
cozinha = bc.compile()

class Salao(TypedDict):
    pedido: str
    prato: str

bp = StateGraph(Salao)
bp.add_node('cozinha', cozinha)   # compilado vira NÓ
bp.add_edge(START, 'cozinha'); bp.add_edge('cozinha', END)
for chunk in bp.compile().stream({'pedido': 'pastel', 'prato': ''}, stream_mode='updates'):
    print(chunk)
print()
print('Leitura: o nó-filho devolveu pedido (eco) + prato (novo).')

## 2. Conceito: ler filtra, devolver ecoa

- **Leitura**: o filho recebe o estado do pai **filtrado pelo schema dele**
  (só as chaves que ele declara). Chave do pai fora do schema do filho é
  invisível lá dentro.
- **Escrita**: o filho devolve o **estado final completo dele** — canais que
  ele nem tocou voltam junto (eco da entrada).
- **Consequência**: dois filhos paralelos que leem a mesma chave do pai
  **escrevem ela de volta na mesma etapa** → `InvalidUpdateError` (Aula 08,
  agora entre grafos). Três saídas, da pior para a melhor (seções 3–5).

In [ ]:
# CÉLULA 2 — eco duplo quebra (rode e leia o erro)
from langgraph.errors import InvalidUpdateError

class Bar(TypedDict):
    pedido: str
    bebida: str

bb = StateGraph(Bar)
bb.add_node('servir', lambda s: {'bebida': f"bebida({s['pedido']})"})
bb.add_edge(START, 'servir'); bb.add_edge('servir', END)
bar = bb.compile()

class Salao2(TypedDict):
    pedido: str
    prato: str
    bebida: str

bp2 = StateGraph(Salao2)
bp2.add_node('cozinha', cozinha)
bp2.add_node('bar', bar)
bp2.add_edge(START, 'cozinha'); bp2.add_edge(START, 'bar')
bp2.add_edge('cozinha', END); bp2.add_edge('bar', END)
try:
    bp2.compile().invoke({'pedido': 'pastel', 'prato': '', 'bebida': ''})
    print('rodou?! (não deveria)')
except InvalidUpdateError as erro:
    print('InvalidUpdateError:', str(erro).splitlines()[0])
    print()
    print('Leitura: cozinha e bar ecoaram pedido na mesma etapa.')

## 3. Dissecando o eco (por que ele existe)

- O filho é um grafo completo: recebe input, executa, devolve **output final**.
  O pai não distingue 'canal lido' de 'canal escrito' — recebe tudo.
- Com **1 filho**, o eco é inofensivo (sobrescreve igual). Com **2+ em paralelo**
  na mesma chave, vira escrita concorrente.
- Saída 1 (curso): reducer na chave comum — evita o crash, mas o eco **duplica**
  (`'pastel'` + ecos → `'pastelpastelpastel'`). Funciona, corrompe.
- Saída 2 (limpa): `output_schema` no filho — só as chaves declaradas voltam.
  É o `output_schema` da Aula 09, agora como interface do filho.

## 4. Variações que quebram (todas executadas e verificadas)

### 4a. Reducer na chave comum: salva, mas duplica

Com `Annotated[str, add]`, o crash some — e o `pedido` vira
`'pastelpastelpastel'` (original + 2 ecos). Prova de que 'não quebrou'
não é 'está certo'.

In [ ]:
# CÉLULA 3 — reducer duplica o eco (rode e confira)
from operator import add
from typing import Annotated

class SalaoAdd(TypedDict):
    pedido: Annotated[str, add]
    prato: str
    bebida: str

bp3 = StateGraph(SalaoAdd)
bp3.add_node('cozinha', cozinha)
bp3.add_node('bar', bar)
bp3.add_edge(START, 'cozinha'); bp3.add_edge(START, 'bar')
bp3.add_edge('cozinha', END); bp3.add_edge('bar', END)
out = bp3.compile().invoke({'pedido': 'pastel', 'prato': '', 'bebida': ''})
print('pedido:', repr(out['pedido']))
assert out['pedido'] == 'pastelpastelpastel'
print('OK — sem crash, com corrupção. Reducer não era a resposta aqui.')

### 4b. `output_schema`: escrita cirúrgica de volta

Cada filho declara o que devolve (`SoPrato`, `SoBebida`). O pai recebe só
isso: `pedido` intacto, chave privada (`rascunho`) sem vazar, e o pai fecha
a conta juntando os dois.

In [ ]:
# CÉLULA 4 — output_schema resolve (rode e confira)
class Cozinha2(TypedDict):
    pedido: str
    prato: str
    rascunho: str

class SoPrato(TypedDict):
    prato: str

class SoBebida(TypedDict):
    bebida: str

bc2 = StateGraph(Cozinha2, output_schema=SoPrato)
bc2.add_node('cozinhar', lambda s: {'prato': f"prato({s['pedido']})",
                                   'rascunho': 'anotação interna'})
bc2.add_edge(START, 'cozinhar'); bc2.add_edge('cozinhar', END)

bb2 = StateGraph(Bar, output_schema=SoBebida)
bb2.add_node('servir', lambda s: {'bebida': f"bebida({s['pedido']})"})
bb2.add_edge(START, 'servir'); bb2.add_edge('servir', END)

class Salao3(TypedDict):
    pedido: str
    prato: str
    bebida: str
    conta: str

bp4 = StateGraph(Salao3)
bp4.add_node('cozinha', bc2.compile())
bp4.add_node('bar', bb2.compile())
bp4.add_node('fechar', lambda s: {'conta': f"{s['prato']} + {s['bebida']}"})
bp4.add_edge(START, 'cozinha'); bp4.add_edge(START, 'bar')
bp4.add_edge('cozinha', 'fechar'); bp4.add_edge('bar', 'fechar')
bp4.add_edge('fechar', END)
out = bp4.compile().invoke({'pedido': 'pastel', 'prato': '', 'bebida': '', 'conta': ''})
print(out)
assert out['pedido'] == 'pastel' and 'rascunho' not in out
assert out['conta'] == 'prato(pastel) + bebida(pastel)'
print('OK — cirúrgico: pedido intacto, rascunho contido, conta fechada.')

## 5. Erros clássicos

1. **Achar que o filho devolve 'só o novo'.** Ele devolve o estado final
   completo (eco incluso). Com 1 filho, inofensivo; com 2+, concorrente.
2. **Reducer na chave comum como 'solução'.** Evita o crash duplicando o eco
   (CÉLULA 3). Reducer mescla valores legítimos, não ecos.
3. **Esquecer `output_schema` no filho compartilhado.** Interface sem contrato
   vaza tudo — inclusive rascunho interno (CÉLULA 4 mostra o contrário).
4. **Pai sem a chave que o filho lê.** O filho enxerga só o que declara; se o
   pai não tem `pedido`, o filho lê vazio/`KeyError` (Aula 09, seção 4b).
5. **Sub-grafo sem `compile()`.** `add_node` aceita runnable compilado — grafo
   cru (`StateGraph` sem `compile`) não é nó válido.
6. **Aninhar sem necessidade.** Sub-grafo organiza times/equipes; para 2 nós
   simples, função + aresta basta. Modularidade tem custo de leitura.

## 6. Exercício guiado — salão com 2 cozinhas e conta

Situação: o salão recebe `pedido`, chama `cozinha` (prato) e `bar` (bebida)
**em paralelo com `output_schema`**, e `fechar` monta a `conta`.
Valide: `pedido` intacto, sem `rascunho`, conta fechada.

In [ ]:
# CÉLULA 5 — exercício guiado (monte o salão completo)
salao = bp4.compile()   # o grafo da CÉLULA 4, pronto
out = salao.invoke({'pedido': 'pastel', 'prato': '', 'bebida': '', 'conta': ''})
print(out)
assert out == {'pedido': 'pastel', 'prato': 'prato(pastel)',
               'bebida': 'bebida(pastel)', 'conta': 'prato(pastel) + bebida(pastel)'}, out
print('OK — reuse o grafo pronto: composição também é reuso.')

## 7. Desafio — terceiro filho com segredo próprio

Adicione a `sobremesa` (lê `pedido`, escreve `doce`, tem `rascunho2` interno)
em paralelo com cozinha e bar, com `output_schema` próprio, e inclua o doce
na `conta` do `fechar`:

- Valide que os 3 ecos não conflitam (`pedido` intacto, 1 execução do `fechar`).
- Valide que nem `rascunho` nem `rascunho2` vazam.
- **Pegadinha:** `fechar` agora tem 3 chegadas — simétricas (mesma etapa),
  então roda 1x (Aula 16). Se um ramo ganhar etapa extra, volta a disparar 2x.

## 8. Checklist + próxima aula

**Você entendeu se consegue responder sem olhar:**
- [ ] O que o filho lê do pai — e o que ele devolve (tudo ou só o novo)?
- [ ] Por que 2 filhos paralelos quebram na chave comum — e as 2 saídas?
- [ ] O que `output_schema` muda na escrita de volta (e no vazamento)?
- [ ] Reducer na chave comum: por que 'funciona' e por que está errado?
- [ ] Quando sub-grafo vale o custo — e quando função + aresta basta?
- [ ] No seu avaliador, que etapas virariam sub-grafos (quais interfaces)?

**Próxima aula (18 — Sub-grafos no LangSmith):** enxergar pai e filhos no
trace — quem chamou quem, quanto tempo cada um levou e onde o erro nasceu.